# CryptoAdvisor Pro — Propozycja Projektu

---

## 1. Nazwa projektu

**CryptoAdvisor Pro**

---

## 2. Krótki opis projektu

Webowa aplikacja analityczno-doradcza do tradingu kryptowalut, zbudowana w **Pythonie + Streamlit**, podpięta pod **Binance Public API** (bez potrzeby posiadania klucza API dla danych rynkowych).

Aplikacja w czasie rzeczywistym analizuje rynek kryptowalut, oblicza wskaźniki techniczne i generuje **sygnały tradingowe** (kupno / sprzedaż / neutralny) wraz z dokładnymi poziomami:
- **Entry** (cena wejścia)
- **Stop Loss**
- **Take Profit 1 / 2 / 3**
- **Risk:Reward ratio**

Projekt skupia się na dwóch głównych strategiach:
- **Scalping** (timeframes: 1m, 3m, 5m)
- **Swing Trading** (timeframes: 1h, 4h, 1d)

---

## 3. Problem, który rozwiązuje

Trader musi jednocześnie śledzić dziesiątki wskaźników technicznych, wiele par walutowych i różne timeframe'y — to zajmuje dużo czasu i prowadzi do błędów wynikających z nadmiaru informacji.

**CryptoAdvisor Pro** agreguje wszystkie dane w jednym miejscu, automatycznie oblicza wskaźniki i syntetyzuje je w czytelny sygnał z konkretnym score (-100 do +100) i poziomem pewności (LOW / MEDIUM / HIGH / VERY HIGH), umożliwiając szybsze i bardziej świadome decyzje tradingowe.

---

## 4. Użyte technologie i dane

### Technologie
| Warstwa | Technologia |
|---|---|
| Frontend / UI | Streamlit |
| Backend / logika | Python 3.11+ |
| Wykresy | Plotly |
| Wskaźniki techniczne | własna implementacja (NumPy, Pandas) |
| Dane rynkowe | Binance Public REST API |
| Cache | Streamlit `@st.cache_data` (TTL 15–60s) |

### Źródła danych
- **`api.binance.com/api/v3`** — ceny spot, świece OHLCV, order book, ticker 24h
- **`fapi.binance.com/fapi/v1`** — funding rate, open interest (futures)
- Dane aktualizowane co 15–60 sekund (cache TTL)
- Brak konieczności posiadania konta Binance dla danych rynkowych

---

## 5. Architektura aplikacji

Architektura opisana jest w pliku `architektura_CryptoAdvisor.png` dołączonym do zgłoszenia.

### Warstwy aplikacji

**Warstwa prezentacji (Streamlit):**
```
app.py
├── pages/dashboard.py    # przegląd rynku, ceny live, watchlist, movers
├── pages/signals.py      # sygnały + skaner wielu par jednocześnie
├── pages/analysis.py     # wykresy OHLCV, order book depth
└── pages/settings.py     # konfiguracja API key, preferencje alertów
```

**Warstwa logiki (Core Engine):**
```
core/
├── binance_client.py  # pobieranie danych z Binance API (z cache)
├── indicators.py      # obliczanie wskaźników technicznych
└── signals.py         # generowanie sygnałów tradingowych
```

---

## 6. Zaimplementowane wskaźniki techniczne

| Kategoria | Wskaźniki |
|---|---|
| Momentum | RSI(7), RSI(14), Williams %R, CCI |
| Trend | EMA(9/21/50/200), SMA(20), MACD |
| Zmienność | Bollinger Bands (z %B i bandwidth), ATR |
| Wolumen | VWAP, OBV, Volume Ratio |
| Stochastic | Stochastic K/D |
| Poziomy | Automatyczna detekcja wsparć i oporów |

---

## 7. System sygnałów

Każdy wskaźnik dodaje lub odejmuje punkty od **score** (zakres -100 do +100).

```python
# Przykładowa logika:
if rsi_7 < 25:
    score += 25   # głęboka wyprzedanie
if macd_crossover_bullish:
    score += 20   # bycze przecięcie MACD
if bb_pct_b < 0.05:
    score += 20   # cena przy dolnym Bollingerze

# Finalna decyzja:
# score >= +30  → BUY
# score <= -30  → SELL
# pozostałe    → NEUTRAL
```

**Poziomy SL/TP** obliczane są na podstawie ATR (Average True Range), co automatycznie dostosowuje je do aktualnej zmienności rynku.

---

## 8. Główne funkcje aplikacji

1. **Dashboard** — ceny live 20+ par, top gainers/losers 24h, watchlist, szybki sygnał
2. **Szczegółowy sygnał** — pełna analiza wybranej pary z listą powodów i ostrzeżeń
3. **Analiza Multi-Timeframe** — jednoczesna analiza 5 timeframe'ów (1m/5m/15m/1h/4h)
4. **Skaner rynku** — automatyczne skanowanie wybranych par w poszukiwaniu sygnałów
5. **Wykres techniczny** — interaktywny OHLCV z nakładkami (BB, EMA, VWAP) + RSI/MACD/Volume
6. **Order Book Depth** — wizualizacja głębokości rynku z sentymentem bid/ask

---

## 9. Potencjalny model biznesowy (SaaS)

Aplikacja jest zaprojektowana z myślą o sprzedaży na **subskrypcję**:

| Plan | Funkcje |
|---|---|
| Free | Dashboard, 3 pary, 1 timeframe |
| Pro | Wszystkie funkcje, skaner, MTF, alerty |
| Elite | Pro + alerty Telegram/webhook, priorytetowe wsparcie |

**Stack deploymentu:** Streamlit Community Cloud / Railway / VPS + `streamlit-authenticator` lub Auth0

---

## 10. Co już zostało zrobione?

Projekt nie jest **zaimplementowany** jako MVP (Minimum Viable Product):

- [] Połączenie z Binance Public API (dane spot + futures)
- [] Silnik wskaźników technicznych (10+ wskaźników)
- [] System sygnałów scalpingowych i swingowych ze scoringiem
- [] 4 strony Streamlit z ciemnym motywem (dark trading theme)
- [] Interaktywne wykresy Plotly (OHLCV + podpanele)
- [] Skaner rynku (wiele par jednocześnie)
- [] Analiza Multi-Timeframe
- [] Order Book z wizualizacją sentymentu
- [] Alerty Telegram / webhook (planowane)
- [] System autentykacji użytkowników (planowane)
- [] Backtest strategii (planowane)

---

## 11. Czego się nauczę podczas projektu?

- Budowania wielostronicowych aplikacji w Streamlit
- Implementacji wskaźników analizy technicznej od podstaw (bez bibliotek taLolib)
- Pracy z REST API (asynchroniczne pobieranie danych, cachowanie)
- Projektowania systemu scoringowego dla sygnałów tradingowych
- Tworzenia interaktywnych wykresów finansowych w Plotly
- Myślenia o architekturze aplikacji (separacja warstw: data / logic / UI)

---

## 12. Potencjalny rozwój projektu

- **Backtesting** — testowanie strategii na danych historycznych
- **Paper Trading** — symulowany trading bez prawdziwych środków
- **Machine Learning** — model ML do predykcji sygnałów (np. XGBoost / LSTM)
- **Alerty push** — Telegram bot, Discord webhook, email
- **Portfolio tracker** — śledzenie portfela przez read-only API Binance
- **Social features** — leaderboard traderów, sharing sygnałów


---
## Demo — pobieranie danych z Binance API

In [ ]:
import requests
import pandas as pd

# Pobieranie świec OHLCV dla BTC/USDT (5 minut, ostatnie 10 świec)
r = requests.get('https://api.binance.com/api/v3/klines', params={
    'symbol': 'BTCUSDT',
    'interval': '5m',
    'limit': 10
})

df = pd.DataFrame(r.json(), columns=[
    'open_time','open','high','low','close','volume',
    'close_time','quote_volume','trades','taker_buy_base','taker_buy_quote','ignore'
])

for col in ['open','high','low','close','volume']:
    df[col] = df[col].astype(float)

df['open_time'] = pd.to_datetime(df['open_time'], unit='ms')
print(df[['open_time','open','high','low','close','volume']].to_string(index=False))

## Demo — obliczanie RSI

In [ ]:
import numpy as np

def rsi(series, period=14):
    """Relative Strength Index — własna implementacja"""
    delta = series.diff()
    gain  = delta.where(delta > 0, 0.0)
    loss  = -delta.where(delta < 0, 0.0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# Pobieramy więcej danych żeby RSI miał sens
r2 = requests.get('https://api.binance.com/api/v3/klines', params={
    'symbol': 'BTCUSDT', 'interval': '5m', 'limit': 50
})
df2 = pd.DataFrame(r2.json())
df2[4] = df2[4].astype(float)  # close

rsi_values = rsi(df2[4], 14)
current_rsi = rsi_values.iloc[-1]

print(f'Aktualny RSI(14) dla BTC/USDT 5m: {current_rsi:.2f}')
if current_rsi < 30:
    print('=> SYGNAŁ: wyprzedanie (potencjalny LONG)')
elif current_rsi > 70:
    print('=> SYGNAŁ: wykupienie (potencjalny SHORT)')
else:
    print('=> SYGNAŁ: neutralny')

## Demo — uruchomienie aplikacji

```bash
# Instalacja zależności
pip install streamlit pandas numpy plotly requests

# Uruchomienie aplikacji
streamlit run app.py
# Aplikacja otworzy się na: http://localhost:8501
```